# Yak: MLM pretraining (Colab)

Pretrains the Yak encoder (d_model 256, 4 heads, 4 layers, d_ff 1024, sinusoidal positions, tied output head) with masked language modeling, using the BPE tokenizer in `Tokenizer/TokenFile/bpe.json`.

**Before you run anything**
1. Runtime > Change runtime type > GPU (a T4 is enough).
2. Run the cells top to bottom. Data prep and every checkpoint live on Google Drive under `MyDrive/yak`, so if Colab disconnects, re-run the cells and training resumes from the last checkpoint.

**What the cells do**
1. Setup: finds the Yak repo root (works if this notebook sits in `Model/`, anywhere in the repo, or outside it, in which case it clones `feature/mlm`) and puts the root on `sys.path` so `from Model.MlmModel import MlmModel` works.
2. Config: every hyperparameter in one place.
3. Tokenizer: loads the BPE file and builds a fast encoder (the repo encoder is far too slow for a whole corpus), checked against the repo encoder.
4. Data: tokenizes WikiText-103 once and caches the ids on Drive.
5. Model and sanity checks: parameter count, starting loss, masking rate, overfit-one-batch.
6. Training with resume, validation, best-checkpoint saving.
7. Checks: reload the best checkpoint and compare validation loss, fill-the-mask samples, loss curves.

In [ ]:
import os, sys, math, time, json, re, copy, random
import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/gowthambalachandhiran/Yak.git"
BRANCH   = "feature/mlm"
WORK_DIR = "/content/drive/MyDrive/yak"      # tokenized data + checkpoints persist here
os.makedirs(WORK_DIR, exist_ok=True)

def find_repo_root(start):
    # Walk up from `start` until a folder containing both Model/ and Tokenizer/ is found.
    p = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(p, "Model")) and os.path.isdir(os.path.join(p, "Tokenizer")):
            return p
        if os.path.dirname(p) == p:
            return None
        p = os.path.dirname(p)

REPO_DIR = find_repo_root(os.getcwd())            # case 1: notebook is running inside the repo (e.g. in Model/)
if REPO_DIR is None and os.path.isdir("/content/Yak"):
    REPO_DIR = "/content/Yak"                      # case 2: already cloned earlier in this session
if REPO_DIR is None:                               # case 3: fresh runtime, clone it
    REPO_DIR = "/content/Yak"
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}   # fails loudly instead of overwriting local edits

!pip install -q datasets tqdm

# The repo root (not Model/) must be on sys.path: MlmModel does `from Model.LayerOne... import ...`
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
import Model.MlmModel                              # fail early here if the import path is wrong
print("repo root:", REPO_DIR)

In [ ]:
CFG = dict(
    # model (matches README)
    d_model=256, nhead=4, num_layers=4, dropout=0.1, max_seq_len=2048,
    # data / batching
    seq_len=512,              # pretraining length; raise to 2048 for a final phase (see last cell)
    tokens_per_step=32768,    # tokens per optimizer step, independent of seq_len
    micro_bs=64,              # sequences per forward pass; lower it if you hit out-of-memory
    mask_prob=0.15,
    # optimisation
    lr=6e-4, min_lr_frac=0.1, warmup_steps=500, max_steps=12000,
    weight_decay=0.01, betas=(0.9, 0.98), grad_clip=1.0,
    # bookkeeping
    log_every=50, eval_every=500, ckpt_every=500, eval_seqs=256,
    seed=1337,
)
RESUME = True

In [ ]:
from Tokenizer.Tokenizer import BPE

bpe = BPE(vocab_size=30000)
bpe.load(os.path.join(REPO_DIR, "Tokenizer/TokenFile/bpe.json"))

# The saved file has 30,000 vocabulary entries PLUS 5 special tokens. The model needs all of them.
VOCAB_SIZE = len(bpe.tok2id)
assert max(bpe.tok2id.values()) + 1 == VOCAB_SIZE, "token ids are not contiguous"
assert VOCAB_SIZE < 65536, "ids must fit in uint16 for the cached arrays"
PAD, UNK, CLS, SEP, MASK = (bpe.tok2id[t] for t in ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"])
N_SPECIAL = len(bpe.special_tokens)       # ids 0..4 are special and are never masked
print(f"vocab size {VOCAB_SIZE} | merges {len(bpe.merges)} | PAD={PAD} UNK={UNK} CLS={CLS} SEP={SEP} MASK={MASK}")


class FastBPE:
    # Same result as BPE.encode_word, but applies the lowest-ranked merge present
    # instead of running every merge regex over every word, and caches per word.
    def __init__(self, bpe):
        self.rank = {tuple(p): i for i, p in enumerate(bpe.merges)}
        self.tok2id = bpe.tok2id
        self.unk = bpe.tok2id["[UNK]"]
        self.cache = {}

    def _encode(self, word):
        symbols = list(word) + ["</w>"]
        rank = self.rank
        while len(symbols) > 1:
            best, best_rank = None, None
            for i in range(len(symbols) - 1):
                r = rank.get((symbols[i], symbols[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = (symbols[i], symbols[i + 1]), r
            if best is None:
                break
            merged, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                    merged.append(best[0] + best[1]); i += 2
                else:
                    merged.append(symbols[i]); i += 1
            symbols = merged
        return tuple(self.tok2id.get(s, self.unk) for s in symbols)

    def line_ids(self, line):
        out, cache = [], self.cache
        for w in line.split():
            t = cache.get(w)
            if t is None:
                t = cache[w] = self._encode(w)
            out.extend(t)
        return out


fast = FastBPE(bpe)

In [ ]:
from array import array
from tqdm.auto import tqdm
from datasets import load_dataset

DATA_DIR = os.path.join(WORK_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
TRAIN_NPY = os.path.join(DATA_DIR, "train_ids.npy")
VAL_NPY   = os.path.join(DATA_DIR, "val_ids.npy")

ARTICLE_RE = re.compile(r"^ = [^=].* = \n?$")     # WikiText marks a new article with ' = Title = '

def tokenize_rows(rows):
    # One flat uint16 stream. [SEP] separates articles. Training windows add [CLS] at the start.
    out = array("H")
    for line in tqdm(rows, unit="line"):
        if not line.strip():
            continue
        if ARTICLE_RE.match(line) and len(out):
            out.append(SEP)
        out.extend(fast.line_ids(line))
    return np.frombuffer(out, dtype=np.uint16).copy()

if not (os.path.exists(TRAIN_NPY) and os.path.exists(VAL_NPY)):
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

    # Check the fast encoder against the repo encoder on real words before trusting it.
    sample = list({w for line in ds["train"]["text"][:20000] for w in line.split()})
    random.Random(0).shuffle(sample)
    sample = sample[:60]
    for w in sample:
        assert list(fast._encode(w)) == bpe.tokens_to_ids(bpe.encode_word(w)), f"mismatch on {w!r}"
    print(f"fast encoder matches the repo encoder on {len(sample)} words")

    t0 = time.time()
    val_ids = tokenize_rows(ds["validation"]["text"])
    train_ids = tokenize_rows(ds["train"]["text"])
    np.save(VAL_NPY, val_ids); np.save(TRAIN_NPY, train_ids)
    print(f"tokenized in {(time.time() - t0) / 60:.1f} min | unique words cached: {len(fast.cache):,}")

train_ids = np.load(TRAIN_NPY)
val_ids = np.load(VAL_NPY)
print(f"train tokens: {len(train_ids):,} | val tokens: {len(val_ids):,}")
print(f"[UNK] rate: train {np.mean(train_ids == UNK):.4%} | val {np.mean(val_ids == UNK):.4%}")

In [ ]:
device = "cuda"

def sample_batch(data, bs, L, rng):
    # Random windows of L-1 tokens from the stream, with [CLS] prepended.
    starts = rng.integers(0, len(data) - (L - 1), size=bs)
    idx = starts[:, None] + np.arange(L - 1)[None, :]
    body = torch.from_numpy(data[idx].astype(np.int64))
    return torch.cat([torch.full((bs, 1), CLS, dtype=torch.long), body], dim=1)

def val_chunks(data, L, max_seqs=None):
    n = len(data) // (L - 1)
    body = torch.from_numpy(data[: n * (L - 1)].astype(np.int64)).view(n, L - 1)
    chunks = torch.cat([torch.full((n, 1), CLS, dtype=torch.long), body], dim=1)
    return chunks[:max_seqs] if max_seqs else chunks

def mask_tokens(ids, p=0.15, gen=None):
    # Dynamic MLM masking: pick 15% of non-special tokens; of those 80% -> [MASK], 10% -> random token, 10% unchanged.
    dev = ids.device
    sel = (torch.rand(ids.shape, device=dev, generator=gen) < p) & (ids >= N_SPECIAL)
    labels = ids.clone()
    labels[~sel] = -100
    r = torch.rand(ids.shape, device=dev, generator=gen)
    inp = ids.clone()
    inp[sel & (r < 0.8)] = MASK
    rand_pos = sel & (r >= 0.8) & (r < 0.9)
    random_tok = torch.randint(N_SPECIAL, VOCAB_SIZE, ids.shape, device=dev, generator=gen)
    inp[rand_pos] = random_tok[rand_pos]
    return inp, labels

# Baseline to beat: always predict the single most frequent non-special token.
counts = np.bincount(val_ids, minlength=VOCAB_SIZE)
top_tok = int(counts[N_SPECIAL:].argmax()) + N_SPECIAL
BASELINE_ACC = counts[top_tok] / counts[N_SPECIAL:].sum()
print(f"frequency baseline: always predict {bpe.id2tok[top_tok]!r} -> {BASELINE_ACC:.2%} masked accuracy")

In [ ]:
from Model.MlmModel import MlmModel

def build_model():
    return MlmModel(
        vocab_size=VOCAB_SIZE, d_model=CFG["d_model"], nhead=CFG["nhead"],
        num_layers=CFG["num_layers"], max_seq_len=CFG["max_seq_len"],
        dropout=CFG["dropout"], pad_token_id=PAD,
    ).to(device)

model = build_model()
n_total = model.num_parameters()
n_body = model.num_parameters(exclude_embedding=True)
print(f"total parameters: {n_total / 1e6:.2f}M | excluding embedding: {n_body / 1e6:.2f}M (embedding {(n_total - n_body) / 1e6:.2f}M)")
assert 3.0e6 < n_body < 3.4e6, "non-embedding parameter count is off the expected ~3.16M"

In [ ]:
# Sanity check 1: starting loss should be close to ln(vocab_size).
vc = val_chunks(val_ids, CFG["seq_len"], max_seqs=32).to(device)
gen = torch.Generator(device=device); gen.manual_seed(0)
inp, labels = mask_tokens(vc, CFG["mask_prob"], gen)
model.eval()
with torch.no_grad():
    loss0, _, _ = model(inp, labels)
print(f"initial loss {loss0.item():.3f} vs ln(V) = {math.log(VOCAB_SIZE):.3f}")
assert abs(loss0.item() - math.log(VOCAB_SIZE)) < 1.0, "initial loss is far from ln(V); check init/tying"

# Sanity check 2: measured masking rate over non-special tokens.
rate = (labels != -100).sum().item() / (vc >= N_SPECIAL).sum().item()
frac_mask = ((inp == MASK) & (labels != -100)).sum().item() / (labels != -100).sum().item()
print(f"masking rate {rate:.3%} (target 15%) | share of selected turned into [MASK] {frac_mask:.1%} (target 80%)")
assert 0.13 < rate < 0.17

# Sanity check 3: the model must be able to overfit one small batch.
tiny = copy.deepcopy(model).float().train()
for m in tiny.modules():
    if isinstance(m, torch.nn.Dropout):
        m.p = 0.0
topt = torch.optim.AdamW(tiny.parameters(), lr=1e-3)
x = vc[:8, :128].contiguous()
gen.manual_seed(1)
xi, xl = mask_tokens(x, 0.15, gen)
first = None
for i in range(200):
    l, _, _ = tiny(xi, xl)
    topt.zero_grad(); l.backward(); topt.step()
    first = l.item() if first is None else first
print(f"overfit test: loss {first:.3f} -> {l.item():.3f}")
assert l.item() < 0.5 * first, "model cannot overfit a single batch; something is broken"
del tiny, topt
torch.cuda.empty_cache()
print("sanity checks passed")

In [ ]:
use_bf16 = torch.cuda.get_device_capability()[0] >= 8       # T4 has no bf16; it uses fp16 + loss scaling
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=not use_bf16)

L = CFG["seq_len"]
accum = max(1, CFG["tokens_per_step"] // (CFG["micro_bs"] * L))
tokens_per_step = accum * CFG["micro_bs"] * L
print(f"precision {amp_dtype} | seq_len {L} | micro_bs {CFG['micro_bs']} x accum {accum} = {tokens_per_step:,} tokens/step")

model = build_model()
decay, no_decay = [], []
for name, p in model.named_parameters():
    (decay if (p.dim() >= 2 and "tok_emb" not in name) else no_decay).append(p)
opt = torch.optim.AdamW(
    [{"params": decay, "weight_decay": CFG["weight_decay"]}, {"params": no_decay, "weight_decay": 0.0}],
    lr=CFG["lr"], betas=CFG["betas"], fused=True,
)

def lr_at(step):
    lr, lo, w, T = CFG["lr"], CFG["lr"] * CFG["min_lr_frac"], CFG["warmup_steps"], CFG["max_steps"]
    if step < w:
        return lr * (step + 1) / w
    prog = min(1.0, (step - w) / max(1, T - w))
    return lo + 0.5 * (lr - lo) * (1 + math.cos(math.pi * prog))

VAL = val_chunks(val_ids, L, max_seqs=CFG["eval_seqs"])

@torch.no_grad()
def evaluate(m, chunks, bs):
    m.eval()
    g = torch.Generator(device=device); g.manual_seed(0)      # same masks every time -> comparable numbers
    tl = tc = tn = 0
    for i in range(0, len(chunks), bs):
        ids = chunks[i:i + bs].to(device)
        inp, lab = mask_tokens(ids, CFG["mask_prob"], g)
        with torch.autocast("cuda", dtype=amp_dtype):
            loss, logits, targets = m(inp, lab)
        k = targets.numel()
        tl += loss.item() * k; tc += (logits.argmax(-1) == targets).sum().item(); tn += k
    m.train()
    return tl / tn, tc / tn

CKPT_DIR = os.path.join(WORK_DIR, "ckpt"); os.makedirs(CKPT_DIR, exist_ok=True)
LAST, BEST = os.path.join(CKPT_DIR, "last.pt"), os.path.join(CKPT_DIR, "best.pt")

def save_ckpt(path, step, best_val, history):
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                "step": step, "best_val": best_val, "history": history, "cfg": CFG,
                "vocab_size": VOCAB_SIZE}, tmp)
    os.replace(tmp, path)           # atomic, so a disconnect cannot leave a half-written file

start_step, best_val, history = 0, float("inf"), []
if RESUME and os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
    start_step, best_val, history = ck["step"], ck["best_val"], ck["history"]
    print(f"resumed from step {start_step} (best val loss so far {best_val:.4f})")

In [ ]:
model.train()
t_start, tok_seen, run_loss = time.time(), 0, torch.zeros((), device=device)

for step in range(start_step, CFG["max_steps"]):
    lr_now = lr_at(step)
    for g in opt.param_groups:
        g["lr"] = lr_now

    for micro in range(accum):
        rng = np.random.default_rng([CFG["seed"], step, micro])      # deterministic per step -> exact resume
        ids = sample_batch(train_ids, CFG["micro_bs"], L, rng).to(device, non_blocking=True)
        inp, lab = mask_tokens(ids, CFG["mask_prob"])
        with torch.autocast("cuda", dtype=amp_dtype):
            loss, _, _ = model(inp, lab)
        scaler.scale(loss / accum).backward()
        run_loss += loss.detach() / accum
    scaler.unscale_(opt)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    tok_seen += tokens_per_step
    done = step + 1

    if done % CFG["log_every"] == 0:
        avg = run_loss.item() / CFG["log_every"]; run_loss.zero_()
        if not math.isfinite(avg):
            raise RuntimeError(f"loss is {avg} at step {done}; lower lr or check the data")
        el = time.time() - t_start
        eta = (CFG["max_steps"] - done) * el / max(1, done - start_step)
        history.append({"step": done, "train_loss": avg, "lr": lr_now})
        print(f"step {done:6d}/{CFG['max_steps']} | loss {avg:.4f} | lr {lr_now:.2e} | gnorm {float(gnorm):.2f} "
              f"| {tok_seen / el / 1e3:.0f}k tok/s | ETA {eta / 60:.0f} min")

    if done % CFG["eval_every"] == 0 or done == CFG["max_steps"]:
        vl, va = evaluate(model, VAL, CFG["micro_bs"])
        history.append({"step": done, "val_loss": vl, "val_acc": va})
        flag = ""
        if vl < best_val:
            best_val = vl; save_ckpt(BEST, done, best_val, history); flag = "  (new best)"
        print(f"  VAL step {done}: loss {vl:.4f} | masked acc {va:.2%} (baseline {BASELINE_ACC:.2%}){flag}")

    if done % CFG["ckpt_every"] == 0 or done == CFG["max_steps"]:
        save_ckpt(LAST, done, best_val, history)

print("training finished")

In [ ]:
# Done-when check: a reloaded checkpoint must reproduce its validation loss.
ck = torch.load(BEST, map_location=device)
reloaded = build_model()
reloaded.load_state_dict(ck["model"])
vl, va = evaluate(reloaded, VAL, CFG["micro_bs"])
print(f"stored best val loss {ck['best_val']:.4f} | reloaded {vl:.4f} | masked acc {va:.2%} vs baseline {BASELINE_ACC:.2%}")
assert abs(vl - ck["best_val"]) < 5e-3, "reloaded checkpoint does not reproduce the stored validation loss"

In [ ]:
@torch.no_grad()
def fill_mask(m, sentence, k=5):
    # Write masks as [MASK]. WikiText style: spaces around punctuation.
    m.eval()
    parts = sentence.split("[MASK]")
    ids, positions = [CLS], []
    for i, part in enumerate(parts):
        ids += fast.line_ids(part)
        if i < len(parts) - 1:
            positions.append(len(ids)); ids.append(MASK)
    ids.append(SEP)
    logits = m(torch.tensor([ids], device=device))[0]
    print(sentence)
    for pos in positions:
        lg = logits[pos].float(); lg[:N_SPECIAL] = float("-inf")
        top = torch.topk(lg.softmax(-1), k)
        print("   ", ", ".join(f"{bpe.id2tok[i.item()].replace('</w>', '')} ({p:.2f})" for p, i in zip(top.values, top.indices)))

for s in ["The capital of France is [MASK] .",
          "He was born in [MASK] in 1950 .",
          "The [MASK] won the match by two goals .",
          "The river flows into the [MASK] sea ."]:
    fill_mask(reloaded, s)

In [ ]:
import matplotlib.pyplot as plt
tr = [h for h in history if "train_loss" in h]; va_ = [h for h in history if "val_loss" in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([h["step"] for h in tr], [h["train_loss"] for h in tr], label="train")
ax[0].plot([h["step"] for h in va_], [h["val_loss"] for h in va_], label="val", marker="o")
ax[0].set_xlabel("step"); ax[0].set_ylabel("masked LM loss"); ax[0].legend(); ax[0].set_ylim(bottom=0)
ax[1].plot([h["step"] for h in va_], [h["val_acc"] for h in va_], marker="o", label="model")
ax[1].axhline(BASELINE_ACC, color="gray", ls="--", label="frequency baseline")
ax[1].set_xlabel("step"); ax[1].set_ylabel("masked token accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

## After the run

**Bundle for fine-tuning.** Copy `best.pt`, `Tokenizer/TokenFile/bpe.json` and the `CFG` dict together (the checkpoint already stores `cfg` and `vocab_size`).

**Optional final phase at the full 2,048 context.** Most steps run at 512 for speed. To extend:
1. Set `seq_len=2048`, `micro_bs=16`, `max_steps` to the old value plus about 1,500, and `warmup_steps=100`.
2. Re-run the config cell and the cells from "Model and sanity checks" onward. With `RESUME = True` it continues from `last.pt`. The positional table is fixed sin/cos, so nothing has to be learned for the longer length.

**Out of memory?** Lower `micro_bs`. `tokens_per_step` stays constant because gradient accumulation makes up the difference.